코드 system-prompt-example

In [ ]:
from anthropic import Anthropic

client = Anthropic()

system_prompt = """당신은 한국 언론 보도의 정파성 분석 전문가입니다.

다음 원칙을 따르세요:
1. 객관적이고 학술적인 톤을 유지하라.
2. 단계적으로 추론하라(어휘 선택 → 행위자 묘사 → 결론).
3. 결론에 대한 확신도를 명시하라 (높음/중간/낮음).
4. 불확실하면 그렇게 말하라. 추측하지 말라."""

response = client.messages.create(
    model="claude-opus-4-7",
    max_tokens=1024,
    system=system_prompt,
    messages=[
        {"role": "user", "content":
         "다음 기사를 분석하라:\n[기사 본문]"}
    ]
)
print(response.content[0].text)

코드 json-mode-example

In [ ]:
from openai import OpenAI
import json

client = OpenAI()

response = client.chat.completions.create(
    model="gpt-5.5",
    response_format={"type": "json_object"},
    messages=[
        {"role": "system", "content":
         "한국 뉴스 기사를 분석하여 JSON으로 응답하라. 키: "
         "topic (정치/경제/사회/문화/국제), "
         "sentiment (긍정/중립/부정), "
         "key_actors (주요 행위자 배열), "
         "frame (Matthes & Kohring 5프레임), "
         "confidence (0~1)"},
        {"role": "user", "content":
         f"기사: {article_text}"}
    ]
)

result = json.loads(response.choices[0].message.content)
print(f"토픽: {result['topic']}")
print(f"확신도: {result['confidence']}")

코드 rag-basic

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain_anthropic import ChatAnthropic

# 1) 문서 로드와 청크 분할
with open('korean_papers.txt', 'r', encoding='utf-8') as f:
    text = f.read()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,           # 한국어는 영어보다 짧게
    chunk_overlap=50,         # 청크 간 중첩
    separators=["\n\n", "\n", "。", ".", " "]
)
chunks = splitter.create_documents([text])

# 2) 임베딩과 벡터 DB 생성
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
vectordb = Chroma.from_documents(chunks, embeddings,
                                   persist_directory="./chroma_db")

# 3) 검색-생성 체인
llm = ChatAnthropic(model="claude-opus-4-7", temperature=0)
qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectordb.as_retriever(search_kwargs={"k": 5}),
    return_source_documents=True
)

# 4) 질의
query = "Dallas Smythe의 audience commodity 이론과 " \
         "디지털 노동 이론의 관계는?"
result = qa({"query": query})
print(result['result'])
for doc in result['source_documents']:
    print("\n출처:", doc.metadata)
    print(doc.page_content[:200])

코드 system-prompt-design

In [ ]:
당신은 한국 정치 커뮤니케이션 연구 전문가입니다.

## 분석 작업
주어진 기사를 다음 5개 차원에서 분석하여 JSON으로 출력하세요.

## 5개 분석 차원

### 1. 정파성 (politcal_orientation)
- 옵션: "진보", "중도", "보수", "중립"
- 판단 기준: 어휘 선택, 행위자 묘사, 인용 출처
- 5대 일간지 일반적 경향 (참고): 조선·동아·중앙 = 보수, 한겨레 = 진보, 경향 = 중도-진보
  (단, 한 기사가 매체의 일반적 경향과 다를 수 있음)

### 2. 프레임 (frame, Matthes & Kohring 2008)
- problem: 어떤 문제로 정의되는가? (한 문장)
- cause: 원인 진단은? (한 문장)
- moral: 도덕적 평가는? 누가 옳고 그른가? (한 문장)
- solution: 해결책 권고는? (한 문장)
- main_actors: 주요 행위자 배열

### 3. 핵심 인물·기관 (key_entities)
- persons: 등장 인물명 배열
- organizations: 등장 기관명 배열
- locations: 등장 지명 배열

### 4. 토픽 (topic)
- 옵션: "정치권력", "경제정책", "사회복지", "외교안보", "사법개혁", "미디어개혁", "시민운동", "기타"

### 5. 분석 메타데이터
- confidence: 분석 확신도 (0.0 ~ 1.0)
- reasoning: 정파성 판단의 핵심 근거 (한국어 2~3문장)

## 출력 형식
반드시 유효한 JSON으로만 출력하세요.
다른 설명이나 코드 블록 표시 없이 JSON 객체만 출력하세요.

## 중요 원칙
1. 객관적이고 학술적인 톤을 유지하세요.
2. 단계적으로 추론하세요 (어휘 → 행위자 묘사 → 인용 → 결론).
3. 불확실하면 confidence를 낮게 설정하고 "중립"을 선택하세요.
4. 반어법·풍자에 주의하세요 (특히 사설·칼럼).
5. 한 매체의 한 기사가 그 매체의 일반적 경향과 다를 수 있음을 인정하세요.

코드 analysis-pipeline

In [ ]:
import asyncio
import json
from anthropic import AsyncAnthropic
from typing import Optional
import pandas as pd
from tqdm.asyncio import tqdm_asyncio

client = AsyncAnthropic()

# 위에서 정의한 system_prompt 사용
with open('system_prompt.md', 'r', encoding='utf-8') as f:
    SYSTEM_PROMPT = f.read()

async def analyze_article(article: dict, semaphore) -> Optional[dict]:
    async with semaphore:
        try:
            response = await client.messages.create(
                model="claude-opus-4-7",
                max_tokens=1500,
                system=SYSTEM_PROMPT,
                messages=[{
                    "role": "user",
                    "content":
                    f"매체: {article['media']}\n"
                    f"날짜: {article['date']}\n"
                    f"제목: {article['title']}\n\n"
                    f"본문: {article['body'][:3000]}"
                }],
                temperature=0  # 재현가능성을 위해 0
            )
            result = json.loads(response.content[0].text)
            result['article_id'] = article['id']
            return result
        except Exception as e:
            print(f"Error in {article['id']}: {e}")
            return None

async def main():
    df = pd.read_csv('articles.csv')
    articles = df.to_dict('records')

    # 동시 호출 수 제한 (API 한도 관리)
    semaphore = asyncio.Semaphore(10)

    tasks = [analyze_article(a, semaphore) for a in articles]
    results = await tqdm_asyncio.gather(*tasks)

    # 결과를 DataFrame으로 변환
    valid_results = [r for r in results if r is not None]
    results_df = pd.json_normalize(valid_results)
    results_df.to_csv('analyzed_articles.csv', index=False)
    print(f"완료: {len(valid_results)}/{len(articles)}")

asyncio.run(main())

코드 validation

In [ ]:
import krippendorff
import numpy as np
import pandas as pd

# 200건 샘플의 라벨 로드
llm_labels = pd.read_csv('llm_validation_sample.csv')
human_labels = pd.read_csv('human_validation_sample.csv')

# 정파성 차원의 α 계산 (명목 척도)
# 진보=0, 중도=1, 보수=2, 중립=3로 인코딩
label_map = {'진보': 0, '중도': 1, '보수': 2, '중립': 3}

data = np.array([
    [label_map[x] for x in llm_labels['political_orientation']],
    [label_map[x] for x in human_labels['coder1']],
    [label_map[x] for x in human_labels['coder2']]
])

alpha = krippendorff.alpha(
    reliability_data=data,
    level_of_measurement='nominal'
)
print(f"3-way Krippendorff's α: {alpha:.4f}")

# 부트스트랩 신뢰구간
from sklearn.utils import resample
n = data.shape[1]
alphas = []
for _ in range(1000):
    idx = resample(range(n), n_samples=n)
    alphas.append(krippendorff.alpha(
        reliability_data=data[:, idx],
        level_of_measurement='nominal'
    ))
print(f"95% CI: [{np.percentile(alphas, 2.5):.4f}, "
      f"{np.percentile(alphas, 97.5):.4f}]")